In [ ]:
%load_ext autoreload
%autoreload 2


## 1. Imports


In [ ]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import math
import os
import random

import numpy as np
import torch
from comet_ml import Experiment
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from torch import optim
from tqdm import tqdm

from src.utils.datasets.alae import get_latents
from src.utils.datasets.paired import get_paired_sampler
from src.utils.training.helpers import compute_loss, update_average
from src.utils.plotting.parameters import plot_A_parameters, plot_B_parameters
from src.utils.core.seed import set_seed
from src.utils.samplers.data import DatasetSampler, TensorSampler


In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device


In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.set_default_dtype(dtype)


## 2. Config

EBiEOT-GMM fits a transport map in FFHQ **latent space**. The generative density \(\rho_\theta\) on latents is a normalized probability model (integrates to 1); transport updates conditional \(\pi(y|x)\) via the GMM dual and LSE cost.

Papermill parameters `INPUT_DATA` / `TARGET_DATA` must match the chosen `EXPERIMENT` preset (or pass Hydra overrides).


In [ ]:
# Papermill: EXPERIMENT selects conf/experiment/<name>.yaml.
# Leave INPUT_DATA / TARGET_DATA empty to use values from the experiment preset.
EXPERIMENT = "gmm-alae-adult-children"  # see alias table below
INPUT_DATA = ""  # ADULT | CHILDREN | WOMAN | MAN (optional override)
TARGET_DATA = ""
OVERRIDES: list[str] = [
    # "train.steps_to=500",
    # "ebieot.model.n_potentials=10",
]


def hydra_experiment(display_name: str) -> str:
    aliases = {
        "gmm-alae": "gmm_alae",
        "gmm-alae-adult-children": "gmm_alae_adult_children",
        "gmm-alae-children-adult": "gmm_alae_children_adult",
        "gmm-alae-woman-man": "gmm_alae_woman_man",
        "gmm-alae-alt": "gmm_alae_alt",
    }
    return aliases.get(display_name, display_name.replace("-", "_"))


compose_overrides = [f"experiment={hydra_experiment(EXPERIMENT)}", *OVERRIDES]
if INPUT_DATA:
    compose_overrides.append(f"dataset.input_data={INPUT_DATA}")
if TARGET_DATA:
    compose_overrides.append(f"dataset.target_data={TARGET_DATA}")

CONF_DIR = os.path.abspath(str(REPO_ROOT / "conf"))
with initialize_config_dir(version_base=None, config_dir=CONF_DIR):
    cfg = compose(config_name="config", overrides=compose_overrides)

INPUT_DATA = str(cfg.dataset.input_data)
TARGET_DATA = str(cfg.dataset.target_data)

seed = int(cfg.seed) if cfg.get("seed") is not None else int(cfg.train.seed)
set_seed(seed)
print(f"Direction: {INPUT_DATA} -> {TARGET_DATA}")
print(OmegaConf.to_yaml(cfg.train))


| `EXPERIMENT` | Direction | Notes |
| --- | --- | --- |
| `gmm-alae-adult-children` | ADULT → CHILDREN | legacy `ebieot_gmm_ALAE_old_to_young` |
| `gmm-alae-children-adult` | CHILDREN → ADULT | larger batches, `lr_paired=1e-5` |
| `gmm-alae-woman-man` | WOMAN → MAN | `n_potentials=10` |
| `gmm-alae-alt` | ADULT → CHILDREN | alternate `n_potentials` / cost width |


## 3. Model & data

FFHQ latents live under `datasets/FFHQ/` (see `conf/dataset/alae/ffhq_latents.yaml`). Paired OT tensors are precomputed at `datasets/FFHQ/pairs/{INPUT}->{TARGET}/`.


In [ ]:
ds = cfg.dataset
data_root = REPO_ROOT / str(ds.data_root)

X_train, X_test = get_latents(INPUT_DATA, from_dir=str(data_root), dtype=dtype)
Y_train, Y_test = get_latents(TARGET_DATA, from_dir=str(data_root), dtype=dtype)

X_sampler = TensorSampler(X_train.to(dtype), device=str(device))
Y_sampler = TensorSampler(Y_train.to(dtype), device=str(device))

pairs_dir = data_root / "pairs" / f"{INPUT_DATA}->{TARGET_DATA}"
for name in ("X_train.pt", "Y_train.pt", "X_test.pt", "Y_test.pt"):
    if not (pairs_dir / name).is_file():
        raise FileNotFoundError(f"Missing paired data: {pairs_dir / name}")

X_paired_train = torch.load(pairs_dir / "X_train.pt", map_location=device, weights_only=True).to(dtype)
Y_paired_train = torch.load(pairs_dir / "Y_train.pt", map_location=device, weights_only=True).to(dtype)
X_paired_test = torch.load(pairs_dir / "X_test.pt", map_location=device, weights_only=True).to(dtype)
Y_paired_test = torch.load(pairs_dir / "Y_test.pt", map_location=device, weights_only=True).to(dtype)

P_XY = min(int(ds.P_XY_paired), X_paired_train.shape[0])
Q_X = min(int(ds.Q_X_unpaired), X_train.shape[0])
R_Y = min(int(ds.R_Y_unpaired), Y_train.shape[0])

pd_train_sampler = get_paired_sampler(
    X_paired_train,
    Y_paired_train,
    int(cfg.train.paired_batch_size),
    P_XY,
    str(device),
)

if bool(ds.get("unpaired_use_full_train", False)):
    usd_sampler = DatasetSampler(X_train, device=str(device))
    utd_sampler = DatasetSampler(Y_train, device=str(device))
else:
    if Q_X > 0:
        usd_sampler = DatasetSampler(X_sampler.sample(Q_X), device=str(device))
    else:
        usd_sampler = DatasetSampler(X_paired_train, device=str(device))
    if R_Y > 0:
        utd_sampler = DatasetSampler(Y_sampler.sample(R_Y), device=str(device))
    else:
        utd_sampler = DatasetSampler(Y_paired_train, device=str(device))

model = build_gmm_model(cfg, device)
model.init_a_by_samples(Y_sampler.sample(int(cfg.ebieot.model.n_potentials)))

model_copy = None
if cfg.train.ema_update:
    model_copy = build_gmm_model(cfg, device)
    model_copy.load_state_dict(model.state_dict())


## 4. Optimizers


In [ ]:
def _adam(params, opt_cfg):
    kwargs = {
        "lr": float(opt_cfg.lr),
        "betas": tuple(float(b) for b in opt_cfg.betas),
    }
    if opt_cfg.get("weight_decay") is not None:
        kwargs["weight_decay"] = float(opt_cfg.weight_decay)
    return optim.Adam(params, **kwargs)


unpaired_cfg = cfg.train.optimizer.unpaired
paired_cfg = cfg.train.optimizer.paired

unpaired_params = [model._log_w_n, model._a_n, model._log_A_n]
D_opt_unpaired = _adam(unpaired_params, unpaired_cfg)
D_opt_paired = _adam(model.cost.parameters(), paired_cfg)


In [ ]:
train_cfg = cfg.train
gm = cfg.ebieot.model
cost = cfg.ebieot.cost
EXP_NAME = (
    f"EBiEOT-GMM-ALAE-{INPUT_DATA}-to-{TARGET_DATA}-"
    f"P{P_XY}_Q{Q_X}_R{R_Y}_"
    f"N{gm.n_potentials}_M{cost.m_potentials}_"
    f"lr_p{float(paired_cfg.lr):.0e}_lr_u{float(unpaired_cfg.lr):.0e}"
)
OUTPUT_PATH = REPO_ROOT / "checkpoints" / EXP_NAME
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

if int(train_cfg.steps_from) > 0:
    D_opt_unpaired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_from}.pt", map_location=device)
    )
    D_opt_paired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_from}.pt", map_location=device)
    )


## 5. Training


In [ ]:
starting_points = X_test[:3]
num_starting_points_paired = min(5, P_XY)
if num_starting_points_paired > 0:
    indices = random.choices(range(P_XY), k=num_starting_points_paired)
    starting_points_paired = X_paired_train[indices]
    ending_points_paired = Y_paired_train[indices]
else:
    starting_points_paired = ending_points_paired = None

max_norm = float(train_cfg.gradient_max_norm)
clip_grads = math.isfinite(max_norm)


In [ ]:
experiment = Experiment(project_name="ebieot")
experiment.set_name(EXP_NAME)
experiment.log_parameters(OmegaConf.to_container(cfg, resolve=True))

for step in tqdm(range(int(train_cfg.steps_from), int(train_cfg.steps_to))):
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(int(train_cfg.unpaired_batch_size))
    Y = utd_sampler.sample(int(train_cfg.unpaired_batch_size))
    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(int(train_cfg.paired_batch_size))
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()

    if clip_grads:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        torch.nn.utils.clip_grad_norm_(model.cost.parameters(), max_norm=max_norm)

    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_cfg.ema_update and model_copy is not None:
        update_average(model_copy, model, 0.99)
        model = model_copy

    log_every = int(train_cfg.get("log_every", 0))
    if log_every > 0 and step % log_every == 0:
        experiment.log_metrics(
            {
                "Unpaired loss": D_loss_unpaired.item(),
                "Paired loss": D_loss_paired.item(),
                "Loss": D_loss.item(),
                "Train paired loss": compute_loss(
                    model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train
                ),
                "Test paired loss": compute_loss(
                    model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test
                ),
            },
            step=step,
        )
        if "f_c" in output_unpaired:
            experiment.log_metric("-f^c(x)", -output_unpaired["f_c"].mean().item(), step=step)
        if "f" in output_unpaired:
            experiment.log_metric("-f(y)", -output_unpaired["f"].mean().item(), step=step)
        if "A_n" in output_unpaired:
            experiment.log_metrics(
                {
                    "lam_min(A_n)": float(torch.min(output_unpaired["A_n"])),
                    "lam_max(A_n)": float(torch.max(output_unpaired["A_n"])),
                },
                step=step,
            )

    plot_every = int(train_cfg.get("plot_every", 0))
    if plot_every > 0 and step % plot_every == 0:
        plot_A_parameters(model)
        plot_B_parameters(model.cost, starting_points)
        torch.save(model.state_dict(), OUTPUT_PATH / f"model_{step}.pt")

torch.save(model.state_dict(), OUTPUT_PATH / f"D_{train_cfg.steps_to}.pt")
torch.save(D_opt_paired.state_dict(), OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_to}.pt")
torch.save(D_opt_unpaired.state_dict(), OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_to}.pt")

experiment.end()


## 6. ALAE image decode (optional)

To visualize decoded faces, add the vendored ALAE stack to `sys.path` (e.g. `notebooks/ALAE/`) and load the FFHQ decoder checkpoint. This section is intentionally omitted from the default training path.
